In [1]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("C:/Users/Administrator/PythonProjects/abfluss_queich")

In [2]:
from pathlib import Path

from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup, Tag
import pandas as pd

from utils.logger import logger
from jobs.temp.stations import STATIONS

ModuleNotFoundError: No module named 'utils'

In [4]:
TEMP_URL = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/10_minutes/air_temperature/now/"

In [51]:
def get_upload_time(html_tag: Tag) -> pd.Timestamp | None:

    tail = str(html_tag.next_sibling)
    
    if tail is None:
        return None
    
    parts = tail.strip().split()
    
    upload_time = pd.to_datetime(
        f"{parts[0]} {parts[1]}",
        dayfirst=True
    )
    
    return upload_time


def get_station_id(href: str) -> str:
    
    _, _, station_id, _ = href.split("_", maxsplit=3)
    
    return station_id


def fetch_temp_metadata(
    url: str | Path,
    station_ids: list[str],
    ) -> pd.DataFrame:
    try:
        response = requests.get(str(url), timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        
        
        rows = []

        for a in soup.find_all("a"):

            # Get filename
            href = str(a.get("href"))
            
            if not href:
                continue

            if not href.endswith(".zip"):
                continue
            
            if all(station not in href for station in station_ids):
                continue
            
            
            station_id = get_station_id(href=href)
            upload_time = get_upload_time(html_tag=a)
            
            if upload_time is None:
                continue
            
            
            rows.append({
                        "filename": href,
                        "station_id": station_id,
                        "datetime_upload": upload_time,
                    })

        return pd.DataFrame(rows)
    
    
    except requests.RequestException as e:
        logger.exception("Failed to fetch icon metadata from %s", url)
        raise
    
    
def download_radolan_file(
    root_url: str,
    file_name: str,
    output_dir: str
    ) -> None:
    
    file_url = urljoin(root_url, file_name)

    output_path = Path(output_dir) / file_name
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    if output_path.exists():
        logger.info("File %s yet exists, skip download", file_name)
        return
        
    try:
        logger.info("Downloading %s", file_name)    
        
        with requests.get(file_url, timeout=30, stream=True) as r:
                    
            r.raise_for_status()

            with open(output_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        
        logger.info("Saved file to %s", output_path)
        
        
    except requests.RequestException:
        logger.exception("Failed downloading file: %s", file_url)
        raise
    
    except OSError:
        logger.exception("Failed writing file: %s", output_path)
        raise

In [52]:
df = fetch_temp_metadata(url=TEMP_URL, station_ids=STATIONS)

In [53]:
df

,filename,station_id,datetime_upload
0,10minutenwerte_TU_00377_now.zip,00377,2026-05-31 20:20:00
1,10minutenwerte_TU_02486_now.zip,02486,2026-05-31 20:20:00
2,10minutenwerte_TU_03939_now.zip,03939,2026-05-31 20:20:01
3,10minutenwerte_TU_05426_now.zip,05426,2026-05-31 20:20:01
